# Bài tập 1

In [1]:
# -*- coding: utf-8 -*-
"""Bài tập 1: Tự viết thuật toán KNN bằng NumPy không dùng thư viện scikit-learn.

So sánh kết quả Accuracy với KNeighborsClassifier của scikit-learn trên bộ hai vầng trăng.
"""

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier


def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    """Dự đoán nhãn cho một mẫu mới x_moi dựa trên thuật toán K láng giềng gần nhất.

    Tham số:
        X_hoc: np.ndarray chứa các mẫu huấn luyện (kích thước N x d)
        y_hoc: np.ndarray chứa nhãn tương ứng (kích thước N,)
        x_moi: np.ndarray chứa tọa độ mẫu mới cần dự đoán (kích thước d,)
        k: số lượng láng giềng gần nhất xem xét

    Trả về:
        nhan_du_doan: nhãn chiếm đa số trong k láng giềng gần nhất (0 hoặc 1)
    """
    # 1. Tính khoảng cách Euclid từ x_moi tới tất cả các điểm trong X_hoc
    # Áp dụng cơ chế vector hóa (broadcasting): (N, d) - (d,) -> (N, d)
    khoang_cach = np.sqrt(np.sum((X_hoc - x_moi) ** 2, axis=1))

    # 2. Tìm chỉ số của k láng giềng có khoảng cách nhỏ nhất
    chi_so_k_gan_nhat = np.argsort(khoang_cach)[:k]

    # 3. Lấy nhãn của k láng giềng tương ứng
    nhan_k_gan_nhat = y_hoc[chi_so_k_gan_nhat]

    # 4. Bỏ phiếu đa số:
    # Với bài toán 2 lớp nhãn {0, 1}, số phiếu lớp 1 chính là tổng các nhãn
    so_phieu_lop_1 = np.sum(nhan_k_gan_nhat == 1)
    so_phieu_lop_0 = k - so_phieu_lop_1

    return 1 if so_phieu_lop_1 > so_phieu_lop_0 else 0


# --- Đọc và chuẩn bị dữ liệu ---
duong_dan_du_lieu = Path("data/hai_vang_trang.csv") if Path("data/hai_vang_trang.csv").is_file() else Path("../data/hai_vang_trang.csv")
df = pd.read_csv(duong_dan_du_lieu)
X = df[["x1", "x2"]].to_numpy(dtype=float)
y = df["nhan"].to_numpy()

# Chia tập học và kiểm tra đồng bộ với bài 3 và bài 4 trong tài liệu
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

k_chon = 13

# --- 1. Dự đoán bằng hàm tự viết ---
y_du_doan_tu_viet = np.array(
    [knn_du_doan(X_tr, y_tr, x_moi, k=k_chon) for x_moi in X_te]
)
acc_tu_viet = accuracy_score(y_te, y_du_doan_tu_viet)

# --- 2. Dự đoán bằng scikit-learn để đối chứng ---
mo_hinh_sklearn = KNeighborsClassifier(n_neighbors=k_chon)
mo_hinh_sklearn.fit(X_tr, y_tr)
acc_sklearn = accuracy_score(y_te, mo_hinh_sklearn.predict(X_te))

# --- In kết quả đối chứng ---
print(f"Accuracy KNN tu viet (k = {k_chon}) : {acc_tu_viet:.4f}")
print(f"Accuracy Scikit-learn (k = {k_chon}) : {acc_sklearn:.4f}")

assert np.isclose(
    acc_tu_viet, acc_sklearn
), "Loi: Ket qua giua hai cach khong khop nhau!"
print("=> Ket qua giua ham tu viet va scikit-learn khop nhau hoan toan.")

# --- In phần giải thích thuật toán ---
giai_thich = """
--- GIẢI THÍCH BẢN CHẤT THUẬT TOÁN KNN ---
1. Bản chất "Học lười" (Lazy Learning): Thuật toán không hề có quá trình tối ưu hóa tham số hay xây dựng phương trình trước. Bước "fit" chỉ đơn giản là lưu trữ toàn bộ ma trận dữ liệu huấn luyện vào bộ nhớ. Mọi tính toán đo đạc chỉ thực sự diễn ra tại thời điểm dự đoán (predict).
2. Tính khoảng cách Euclid bằng Broadcasting: Phép toán `(X_hoc - x_moi) ** 2` tận dụng tính năng Broadcasting của NumPy để trừ vector x_moi với từng hàng của X_hoc đồng thời mà không cần vòng lặp for. Sau đó lấy `sum(axis=1)` rồi khai căn bậc hai để tạo ra mảng khoảng cách từ x_moi tới mọi điểm trong tập học.
3. Chọn láng giềng gần nhất: Hàm `np.argsort` trả về mảng các CHỈ SỐ (index) theo thứ tự khoảng cách tăng dần. Việc lấy `[:k]` giúp trích xuất chính xác k chỉ số của k điểm gần nhất.
4. Bỏ phiếu đa số (Majority Voting): Rút ra k nhãn tương ứng. Với k = 13 là số lẻ, trong bài toán phân loại 2 lớp, việc đếm phiếu luôn đảm bảo tìm được nhãn chiếm đa số tuyệt đối mà không bao giờ xảy ra tình trạng hòa phiếu.
5. Lý do kết quả khớp hoàn toàn với Scikit-learn: Cả hàm tự viết và thư viện (với thiết lập mặc định) đều sử dụng cùng công thức khoảng cách Euclid và cùng quy tắc bỏ phiếu đa số trên cùng tập dữ liệu chia bởi random_state=17.
"""
print(giai_thich)


Accuracy KNN tu viet (k = 13) : 0.9200
Accuracy Scikit-learn (k = 13) : 0.9200
=> Ket qua giua ham tu viet va scikit-learn khop nhau hoan toan.

--- GIẢI THÍCH BẢN CHẤT THUẬT TOÁN KNN ---
1. Bản chất "Học lười" (Lazy Learning): Thuật toán không hề có quá trình tối ưu hóa tham số hay xây dựng phương trình trước. Bước "fit" chỉ đơn giản là lưu trữ toàn bộ ma trận dữ liệu huấn luyện vào bộ nhớ. Mọi tính toán đo đạc chỉ thực sự diễn ra tại thời điểm dự đoán (predict).
2. Tính khoảng cách Euclid bằng Broadcasting: Phép toán `(X_hoc - x_moi) ** 2` tận dụng tính năng Broadcasting của NumPy để trừ vector x_moi với từng hàng của X_hoc đồng thời mà không cần vòng lặp for. Sau đó lấy `sum(axis=1)` rồi khai căn bậc hai để tạo ra mảng khoảng cách từ x_moi tới mọi điểm trong tập học.
3. Chọn láng giềng gần nhất: Hàm `np.argsort` trả về mảng các CHỈ SỐ (index) theo thứ tự khoảng cách tăng dần. Việc lấy `[:k]` giúp trích xuất chính xác k chỉ số của k điểm gần nhất.
4. Bỏ phiếu đa số (Majority Voting):

# Bài tập 2

In [2]:
# -*- coding: utf-8 -*-
"""Bài tập 2: Khoảng cách Manhattan

Sử dụng metric="manhattan" trong KNeighborsClassifier, tìm k tốt nhất qua kiểm định chéo,
sau đó so sánh kết quả trên tập kiểm tra với khoảng cách Euclid.
"""

from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

# --- Chuẩn bị dữ liệu ---
duong_dan_du_lieu = Path("data/hai_vang_trang.csv") if Path("data/hai_vang_trang.csv").is_file() else Path("../data/hai_vang_trang.csv")
df = pd.read_csv(duong_dan_du_lieu)
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# --- Tìm k tốt nhất bằng kiểm định chéo (Manhattan) ---
print("Kiem dinh cheo 5 phan tren tap hoc (Khoang cach Manhattan):")
diem_manhattan = {}
for k in range(1, 52, 2):
    mo_hinh = KNeighborsClassifier(n_neighbors=k, metric="manhattan")
    diem_manhattan[k] = cross_val_score(mo_hinh, X_tr, y_tr, cv=5).mean()

k_tot_manhattan = max(diem_manhattan, key=diem_manhattan.get)
print(f"-> k tot nhat (Manhattan) = {k_tot_manhattan}")

# --- Đánh giá trên tập kiểm tra với k tốt nhất (Manhattan) ---
mh_manhattan = KNeighborsClassifier(n_neighbors=k_tot_manhattan, metric="manhattan").fit(X_tr, y_tr)
acc_manhattan = accuracy_score(y_te, mh_manhattan.predict(X_te))

# --- Trích xuất kết quả khoảng cách Euclid (từ tài liệu thực hành mục 4.2) ---
k_tot_euclid = 13
acc_euclid = 0.9200

# --- In so sánh ---
print("\n--- SO SÁNH HAI CÁCH ĐO KHOẢNG CÁCH ---")
print(f"Khoảng cách Euclid   : k = {k_tot_euclid:2d}, Accuracy = {acc_euclid:.4f}")
print(f"Khoảng cách Manhattan: k = {k_tot_manhattan:2d}, Accuracy = {acc_manhattan:.4f}")

# --- In nhận xét và giải thích ---
phan_nhan_xet = """
--- NHẬN XÉT VÀ GIẢI THÍCH ---
Nhận xét: Đổi cách đo khoảng cách từ Euclid sang Manhattan không làm kết quả (Accuracy) khác đi nhiều trên tập dữ liệu này (từ 0.9200 của Euclid sang 0.9067 của Manhattan, chỉ chênh lệch khoảng 1.33 điểm phần trăm).

Giải thích:
- Cả hai cách đo khoảng cách (đường chim bay của Euclid và đường dích dắc vuông góc của Manhattan) đều bảo toàn rất tốt thứ tự lân cận của các điểm trong không gian 2 chiều có thang đo tương đồng (đặc trưng x1 và x2).
- Sự khác biệt nhỏ chỉ xảy ra cục bộ ở các điểm nằm sát đường biên quyết định cong (decision boundary), khiến tham số k tối ưu có sự dịch chuyển (từ k = 13 lên k = 27), nhưng độ chính xác tổng thể trên tập kiểm tra vẫn duy trì ở mức rất cao và tương đương nhau (~91% - 92%).
"""
print(phan_nhan_xet)


Kiem dinh cheo 5 phan tren tap hoc (Khoang cach Manhattan):


-> k tot nhat (Manhattan) = 27

--- SO SÁNH HAI CÁCH ĐO KHOẢNG CÁCH ---
Khoảng cách Euclid   : k = 13, Accuracy = 0.9200
Khoảng cách Manhattan: k = 27, Accuracy = 0.9067

--- NHẬN XÉT VÀ GIẢI THÍCH ---
Nhận xét: Đổi cách đo khoảng cách từ Euclid sang Manhattan không làm kết quả (Accuracy) khác đi nhiều trên tập dữ liệu này (từ 0.9200 của Euclid sang 0.9067 của Manhattan, chỉ chênh lệch khoảng 1.33 điểm phần trăm).

Giải thích:
- Cả hai cách đo khoảng cách (đường chim bay của Euclid và đường dích dắc vuông góc của Manhattan) đều bảo toàn rất tốt thứ tự lân cận của các điểm trong không gian 2 chiều có thang đo tương đồng (đặc trưng x1 và x2).
- Sự khác biệt nhỏ chỉ xảy ra cục bộ ở các điểm nằm sát đường biên quyết định cong (decision boundary), khiến tham số k tối ưu có sự dịch chuyển (từ k = 13 lên k = 27), nhưng độ chính xác tổng thể trên tập kiểm tra vẫn duy trì ở mức rất cao và tương đương nhau (~91% - 92%).



# Bài tập 3


In [3]:
# -*- coding: utf-8 -*-
"""Bài tập 3: Đổi đơn vị đo và đo lường sự thay đổi của mô hình."""

from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# --- 1. Đọc dữ liệu và đổi đơn vị đo ---
duong_dan_du_lieu = Path("data/sinh_vien_mo_rong.csv") if Path("data/sinh_vien_mo_rong.csv").is_file() else Path("../data/sinh_vien_mo_rong.csv")
df = pd.read_csv(duong_dan_du_lieu)

# Đổi cột chi_tieu từ nghìn đồng sang triệu đồng bằng cách chia cho 1000
df["chi_tieu"] = df["chi_tieu"] / 1000

y = df["qua_mon"]
X = df[["gio_on", "diem_giua_ky", "chi_tieu"]]

# Chia dữ liệu giống bài trước để so sánh công bằng
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# --- 2. Chạy KNN với k=7 KHÔNG chuẩn hoá trên dữ liệu mới ---
mh_khong_chuan_hoa = KNeighborsClassifier(n_neighbors=7)
mh_khong_chuan_hoa.fit(X_tr, y_tr)

acc_moi = accuracy_score(y_te, mh_khong_chuan_hoa.predict(X_te))

# --- 3. In kết quả ---
print(f"Accuracy khi chua doi don vi (muc 3)     : 0.5333")
print(f"Accuracy sau khi doi sang trieu dong     : {acc_moi:.4f}")

# --- 4. In phần giải thích (Chấm nặng phần này) ---
phan_giai_thich = """
--- GIẢI THÍCH KẾT QUẢ BÀI 3 ---
1. Sự thay đổi của Accuracy:
   - Sau khi đổi đơn vị cột chi_tieu sang triệu đồng (không dùng chuẩn hóa), Accuracy tăng vọt từ 0.5333 lên 0.8667.
   - Mức 0.5333 ban đầu thậm chí còn thấp hơn cả mức đoán bừa (0.6000), trong khi mức 0.8667 đã ngang bằng với mô hình có chuẩn hóa ở mục 3.

2. Vì sao chỉ đổi đơn vị mà kết quả lại khác hẳn?
   - Thuật toán KNN tính khoảng cách dựa trên hiệu số tọa độ giữa các điểm. Ở mục 3, cột chi_tieu tính bằng nghìn đồng có độ lệch chuẩn lên tới 1692, gấp hàng trăm lần độ lệch chuẩn của gio_on (7.82) và diem_giua_ky (2.09).
   - Khoảng cách chênh lệch hàng nghìn đơn vị của chi tiêu đã "nuốt chửng" hoàn toàn khoảng cách của hai cột học tập, biến hai cột quan trọng nhất trở nên vô hình.
   - Khi chia cho 1000, độ lệch chuẩn của chi_tieu giảm xuống chỉ còn 1.692, trở về cùng bậc độ lớn với gio_on và diem_giua_ky. Nhờ vậy, khoảng cách Euclid phản ánh hài hòa thông tin của cả 3 đặc trưng, giúp độ chính xác tăng vọt.

3. Nếu có chuẩn hoá (StandardScaler) thì đổi đơn vị có làm kết quả thay đổi không?
   - KHÔNG. Chuẩn hóa Z-score biến đổi từng đặc trưng theo công thức x' = (x - mean) / std.
   - Khi chia toàn bộ cột chi_tieu cho 1000 thì giá trị từng điểm x, giá trị trung bình mean và độ lệch chuẩn std đều giảm đi 1000 lần.
   - Khi đưa vào công thức Z-score, tử số và mẫu số cùng giảm 1000 lần nên hệ số 1000 tự động bị triệt tiêu hoàn toàn:
     [(x/1000) - (mean/1000)] / (std/1000) = (x - mean) / std.
   - Vì vậy, dữ liệu sau khi chuẩn hóa bằng StandardScaler là hoàn toàn đồng nhất dù ban đầu đo bằng nghìn đồng hay triệu đồng, dẫn tới kết quả Accuracy sẽ không đổi.
"""
print(phan_giai_thich)


Accuracy khi chua doi don vi (muc 3)     : 0.5333
Accuracy sau khi doi sang trieu dong     : 0.8667

--- GIẢI THÍCH KẾT QUẢ BÀI 3 ---
1. Sự thay đổi của Accuracy:
   - Sau khi đổi đơn vị cột chi_tieu sang triệu đồng (không dùng chuẩn hóa), Accuracy tăng vọt từ 0.5333 lên 0.8667.
   - Mức 0.5333 ban đầu thậm chí còn thấp hơn cả mức đoán bừa (0.6000), trong khi mức 0.8667 đã ngang bằng với mô hình có chuẩn hóa ở mục 3.

2. Vì sao chỉ đổi đơn vị mà kết quả lại khác hẳn?
   - Thuật toán KNN tính khoảng cách dựa trên hiệu số tọa độ giữa các điểm. Ở mục 3, cột chi_tieu tính bằng nghìn đồng có độ lệch chuẩn lên tới 1692, gấp hàng trăm lần độ lệch chuẩn của gio_on (7.82) và diem_giua_ky (2.09).
   - Khoảng cách chênh lệch hàng nghìn đơn vị của chi tiêu đã "nuốt chửng" hoàn toàn khoảng cách của hai cột học tập, biến hai cột quan trọng nhất trở nên vô hình.
   - Khi chia cho 1000, độ lệch chuẩn của chi_tieu giảm xuống chỉ còn 1.692, trở về cùng bậc độ lớn với gio_on và diem_giua_ky. Nhờ vậy, kho

# Bài tập 4

In [4]:
# -*- coding: utf-8 -*-
"""Bài tập 4: Láng giềng có trọng số.

Thêm tham số weights="distance" vào KNeighborsClassifier và so sánh,
giải thích ảnh hưởng của trọng số đối với số láng giềng k lớn.
"""

from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# --- 1. Đọc và chuẩn bị dữ liệu ---
duong_dan_du_lieu = Path("data/hai_vang_trang.csv") if Path("data/hai_vang_trang.csv").is_file() else Path("../data/hai_vang_trang.csv")
df = pd.read_csv(duong_dan_du_lieu)
X = df[["x1", "x2"]]
y = df["nhan"]

# Giữ nguyên cách chia tập học và tập kiểm tra (tương tự bài e3_chon_k.py)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# --- 2. Chạy mô hình với các giá trị k = 1, 13, 101 và weights='distance' ---
tap_k = [1, 13, 101]

print("Accuracy tren tap hoc va tap kiem tra (weights='distance'):")
print("  k    tap hoc    tap kiem tra")

for k in tap_k:
    mh = KNeighborsClassifier(n_neighbors=k, weights="distance")
    mh.fit(X_tr, y_tr)

    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))

    print(f" {k:3d}    {a_hoc:.4f}      {a_kt:.4f}")

print()

# --- 3. In nhận xét và trả lời câu hỏi ---
phan_nhan_xet_va_giai_thich = """
--- NHẬN XÉT VÀ GIẢI THÍCH ---
1. So sánh với bảng ở mục 4 (khi weights mặc định là 'uniform'):
   - Accuracy trên tập học với weights='distance' LUÔN bằng 1.0000 ở mọi giá trị k. Lý do là vì khi mô hình tự dự đoán cho chính các điểm dữ liệu trong tập học, khoảng cách từ điểm đó tới chính nó bằng 0. Theo công thức 1/khoảng_cách, trọng số của nó sẽ tiến tới vô cực. Mẫu dữ liệu đó luôn tự quyết định nhãn đúng cho chính nó mà không cần quan tâm các láng giềng khác nói gì.
   - Với k = 1, kết quả trên tập kiểm tra giống hệt khi không có trọng số (0.8000) vì chỉ có đúng 1 láng giềng để biểu quyết.

2. Với k = 101, trọng số theo khoảng cách có giúp mô hình bớt cẩu thả không? Vì sao?
   - CÓ, mô hình bớt cẩu thả hơn rất nhiều. Độ chính xác trên tập kiểm tra tăng vọt từ 0.8533 (của mô hình cẩu thả khi dùng uniform) lên 0.9067.
   - Giải thích bản chất: Khi không có trọng số (ai cũng có 1 phiếu bằng nhau), lấy k = 101 (quá lớn) sẽ gom cả rất nhiều mẫu dữ liệu ở xa tít vào biểu quyết. Các mẫu ở xa thường thuộc lớp khác làm loãng phiếu, khiến đường ranh giới bị san phẳng, cắt ngang thô bạo làm mất dạng đường cong mềm mại của hai vầng trăng.
   - Khi có weights="distance", phiếu được chia theo trọng lượng tỷ lệ nghịch với khoảng cách. Dù thu thập tới 101 hàng xóm, nhưng những hàng xóm ở xa thì lá phiếu có trọng số rất nhỏ, gần như không thể lật ngược kết quả. Lá phiếu quyết định vẫn tập trung vào nhóm một vài hàng xóm ở cự ly gần nhất. Do đó, mô hình giữ lại được sự chi tiết của đường biên quyết định, bám sát dữ liệu và "bớt cẩu thả" hơn rất nhiều.
"""
print(phan_nhan_xet_va_giai_thich)


Accuracy tren tap hoc va tap kiem tra (weights='distance'):
  k    tap hoc    tap kiem tra
   1    1.0000      0.8000
  13    1.0000      0.9333
 101    1.0000      0.9067


--- NHẬN XÉT VÀ GIẢI THÍCH ---
1. So sánh với bảng ở mục 4 (khi weights mặc định là 'uniform'):
   - Accuracy trên tập học với weights='distance' LUÔN bằng 1.0000 ở mọi giá trị k. Lý do là vì khi mô hình tự dự đoán cho chính các điểm dữ liệu trong tập học, khoảng cách từ điểm đó tới chính nó bằng 0. Theo công thức 1/khoảng_cách, trọng số của nó sẽ tiến tới vô cực. Mẫu dữ liệu đó luôn tự quyết định nhãn đúng cho chính nó mà không cần quan tâm các láng giềng khác nói gì.
   - Với k = 1, kết quả trên tập kiểm tra giống hệt khi không có trọng số (0.8000) vì chỉ có đúng 1 láng giềng để biểu quyết.

2. Với k = 101, trọng số theo khoảng cách có giúp mô hình bớt cẩu thả không? Vì sao?
   - CÓ, mô hình bớt cẩu thả hơn rất nhiều. Độ chính xác trên tập kiểm tra tăng vọt từ 0.8533 (của mô hình cẩu thả khi dùng uniform) lên 0.9

# Bài tập 5

In [5]:
# -*- coding: utf-8 -*-
"""Bài tập 5: Chọn k cho bài toán hồi quy trên dữ liệu giá nhà.

Sử dụng cross_val_score với neg_mean_absolute_error để tìm k tốt nhất,
sau đó so sánh MAE trên tập kiểm tra với mô hình LinearRegression.
"""

from pathlib import Path
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor

# --- 1. Đọc và chuẩn bị dữ liệu ---
duong_dan_du_lieu = Path("data/gia_nha.csv") if Path("data/gia_nha.csv").is_file() else Path("../data/gia_nha.csv")
df = pd.read_csv(duong_dan_du_lieu)
X = df[["dien_tich"]]
y = df["gia"]

# Cùng cách chia với bài 1 (và mục 6) để so sánh công bằng
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- 2. Tìm k tốt nhất bằng kiểm định chéo trên tập học ---
print("Kiem dinh cheo 5 phan tren tap hoc (tieu chi MAE):")
diem_cv_theo_k = {}

for k in range(1, 21):
    mh_knn = KNeighborsRegressor(n_neighbors=k)
    diem_am = cross_val_score(
        mh_knn, X_tr, y_tr, cv=5, scoring="neg_mean_absolute_error"
    ).mean()
    diem_cv_theo_k[k] = -diem_am

# Tìm k có MAE nhỏ nhất
k_tot_nhat = min(diem_cv_theo_k, key=diem_cv_theo_k.get)
print(f"-> k tot nhat tim duoc: {k_tot_nhat} (MAE tren tap hoc: {diem_cv_theo_k[k_tot_nhat]:.4f})")
print()

# --- 3. Đánh giá trên tập kiểm tra ---
mh_knn_tot = KNeighborsRegressor(n_neighbors=k_tot_nhat).fit(X_tr, y_tr)
mae_knn = mean_absolute_error(y_te, mh_knn_tot.predict(X_te))

mh_duong_thang = LinearRegression().fit(X_tr, y_tr)
mae_duong = mean_absolute_error(y_te, mh_duong_thang.predict(X_te))

print("MAE tren tap kiem tra (ty dong):")
print(f" KNN, k = {k_tot_nhat:<2d}   : {mae_knn:.4f}")
print(f" Duong thang    : {mae_duong:.4f}")
print()

# --- 4. In nhận xét và trả lời câu hỏi ---
phan_nhan_xet_va_tra_loi = """
--- NHẬN XÉT VÀ TRẢ LỜI ---
Câu hỏi: Với k tốt nhất, MAE trên tập kiểm tra có vượt được đường thẳng không?
Trả lời: KHÔNG. Ngay cả khi đã chọn được k tối ưu nhất thông qua kiểm định chéo (k = 2), MAE của KNN trên tập kiểm tra vẫn cao hơn (tức là kém chính xác hơn) so với mô hình đường thẳng (Linear Regression): 0.4867 so với 0.2578 tỷ đồng.

Giải thích:
- Dữ liệu giá nhà có phân bố tăng gần như đường thẳng theo diện tích. Mô hình Hồi quy tuyến tính áp dụng đúng giả định này nên khớp rất hoàn hảo và tận dụng triệt để xu hướng dữ liệu.
- Trong khi đó, KNN đoán giá bằng cách lấy trung bình cục bộ của các điểm xung quanh, nên đường dự đoán có dạng bậc thang gấp khúc thay vì đường thẳng trơn tru.
- Điểm yếu lớn nhất của KNN trong bài toán hồi quy là không thể ngoại suy (extrapolate). Nếu căn hộ mới có diện tích lớn hơn toàn bộ tập huấn luyện, KNN chỉ có thể trả về mức giá của nhóm căn hộ lớn nhất từng thấy. Do đó, MAE của KNN sẽ luôn thua đường thẳng ở bộ dữ liệu mang tính xu hướng tuyến tính rõ ràng này.
"""
print(phan_nhan_xet_va_tra_loi)


Kiem dinh cheo 5 phan tren tap hoc (tieu chi MAE):


-> k tot nhat tim duoc: 2 (MAE tren tap hoc: 0.3498)

MAE tren tap kiem tra (ty dong):
 KNN, k = 2    : 0.4867
 Duong thang    : 0.2578


--- NHẬN XÉT VÀ TRẢ LỜI ---
Câu hỏi: Với k tốt nhất, MAE trên tập kiểm tra có vượt được đường thẳng không?
Trả lời: KHÔNG. Ngay cả khi đã chọn được k tối ưu nhất thông qua kiểm định chéo (k = 2), MAE của KNN trên tập kiểm tra vẫn cao hơn (tức là kém chính xác hơn) so với mô hình đường thẳng (Linear Regression): 0.4867 so với 0.2578 tỷ đồng.

Giải thích:
- Dữ liệu giá nhà có phân bố tăng gần như đường thẳng theo diện tích. Mô hình Hồi quy tuyến tính áp dụng đúng giả định này nên khớp rất hoàn hảo và tận dụng triệt để xu hướng dữ liệu.
- Trong khi đó, KNN đoán giá bằng cách lấy trung bình cục bộ của các điểm xung quanh, nên đường dự đoán có dạng bậc thang gấp khúc thay vì đường thẳng trơn tru.
- Điểm yếu lớn nhất của KNN trong bài toán hồi quy là không thể ngoại suy (extrapolate). Nếu căn hộ mới có diện tích lớn hơn toàn bộ tập huấn luyện, KNN chỉ có t

# Bài tập 6

In [6]:
# -*- coding: utf-8 -*-
"""Bài tập 6: Đo thời gian dự đoán để thấy rõ bản chất 'học lười' của KNN."""

import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

# --- 1. Chuẩn bị công cụ sinh số ngẫu nhiên và các kích thước ---
rng = np.random.default_rng(0)
cac_kich_thuoc = [1000, 10000, 100000]

# Tạo tập kiểm tra cố định gồm 1000 mẫu mới (với 2 đặc trưng)
X_te = rng.random((1000, 2))
ket_qua_do_thoi_gian = []

# --- 2. Vòng lặp đo thời gian cho từng cỡ dữ liệu ---
for kich_thuoc in cac_kich_thuoc:
    X_tr = rng.random((kich_thuoc, 2))
    y_tr = rng.integers(0, 2, size=kich_thuoc)

    mh_knn = KNeighborsClassifier(n_neighbors=5)

    # a. Đo thời gian huấn luyện (.fit)
    bat_dau_fit = time.time()
    mh_knn.fit(X_tr, y_tr)
    thoi_gian_fit = time.time() - bat_dau_fit

    # b. Đo thời gian dự đoán (.predict) cho 1000 mẫu mới
    bat_dau_pred = time.time()
    mh_knn.predict(X_te)
    thoi_gian_pred = time.time() - bat_dau_pred

    ket_qua_do_thoi_gian.append({
        "So_mau_huan_luyen": kich_thuoc,
        "Thoi_gian_fit_(s)": thoi_gian_fit,
        "Thoi_gian_predict_(s)": thoi_gian_pred
    })

# --- 3. In bảng kết quả ---
df_ket_qua = pd.DataFrame(ket_qua_do_thoi_gian)
print("BANG DO THOI GIAN HUAN LUYEN VA DU DOAN")
print(df_ket_qua.to_string(index=False, float_format="{:.6f}".format))
print()

# --- 4. In giải thích (Chấm nặng phần này) ---
phan_giai_thich = """
--- GIẢI THÍCH VÌ SAO KNN ĐƯỢC GỌI LÀ MÔ HÌNH HỌC LƯỜI (LAZY LEARNING) ---

Dựa vào bảng kết quả đo thời gian phía trên, ta dễ dàng quan sát thấy 2 đặc điểm cốt lõi của KNN:

1. Thời gian huấn luyện (.fit) gần như TỨC THÌ và RẤT NHỎ:
   Dù dữ liệu huấn luyện tăng lên gấp 10 lần (từ 1,000 lên 10,000) hay gấp 100 lần (lên 100,000 mẫu), thời gian chạy lệnh .fit vẫn xấp xỉ 0 giây. Lý do là vì ở bước này, KNN hoàn toàn KHÔNG HỌC hay tính toán bất kỳ một phương trình, trọng số hay tham số nào cả. Việc huấn luyện chỉ đơn giản là thao tác "cất" toàn bộ dữ liệu vào bộ nhớ (RAM).

2. Thời gian dự đoán (.predict) RẤT CHẬM và TĂNG TỶ LỆ THUẬN với quy mô dữ liệu:
   Khi dùng 1000 mẫu mới để dự đoán, thời gian predict tăng vọt theo số lượng mẫu huấn luyện. Nguyên nhân là do tại thời điểm dự đoán, KNN bắt đầu "lôi" dữ liệu ra và tính khoảng cách Euclid từ 1000 mẫu mới này tới TOÀN BỘ các mẫu cũ trong tập huấn luyện (lần lượt là 1 nghìn, 10 nghìn, 100 nghìn phép tính x 1000 điểm mới). Dữ liệu càng phình to, nó càng tốn nhiều thời gian đo đạc khoảng cách và tìm hàng xóm.

=> KẾT LUẬN: 
Đúng với tên gọi "Mô hình học lười" (Lazy Learning), KNN trì hoãn hoàn toàn việc học tập và xử lý dữ liệu ở giai đoạn huấn luyện (làm rất nhanh, lười biếng), và đẩy toàn bộ gánh nặng tính toán phức tạp sang giai đoạn dự đoán.
"""
print(phan_giai_thich)


BANG DO THOI GIAN HUAN LUYEN VA DU DOAN
 So_mau_huan_luyen  Thoi_gian_fit_(s)  Thoi_gian_predict_(s)
              1000           0.001004               0.000999
             10000           0.003000               0.002003
            100000           0.033906               0.002000


--- GIẢI THÍCH VÌ SAO KNN ĐƯỢC GỌI LÀ MÔ HÌNH HỌC LƯỜI (LAZY LEARNING) ---

Dựa vào bảng kết quả đo thời gian phía trên, ta dễ dàng quan sát thấy 2 đặc điểm cốt lõi của KNN:

1. Thời gian huấn luyện (.fit) gần như TỨC THÌ và RẤT NHỎ:
   Dù dữ liệu huấn luyện tăng lên gấp 10 lần (từ 1,000 lên 10,000) hay gấp 100 lần (lên 100,000 mẫu), thời gian chạy lệnh .fit vẫn xấp xỉ 0 giây. Lý do là vì ở bước này, KNN hoàn toàn KHÔNG HỌC hay tính toán bất kỳ một phương trình, trọng số hay tham số nào cả. Việc huấn luyện chỉ đơn giản là thao tác "cất" toàn bộ dữ liệu vào bộ nhớ (RAM).

2. Thời gian dự đoán (.predict) RẤT CHẬM và TĂNG TỶ LỆ THUẬN với quy mô dữ liệu:
   Khi dùng 1000 mẫu mới để dự đoán, thời gian predict 